# Model Selection<br><sub>and Information Sharing and Information Criterion</sub>


In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

t = 1000
# 生成 t 个从标准正态分布中抽取的随机变量
x = stats.norm().rvs(t)
# 生成 t 个从自由度为 2 的 t 分布中抽取的随机变量
y = stats.t(df=2).rvs(t)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))

barh, bins, arts = ax[0].hist(x, alpha=0.5)
ax[0].hist(y, bins=bins, alpha=0.5)

support = np.linspace(-10, 10, 100)

for i in range(100):
    ax[1].plot(support, stats.norm(loc=y[i], scale=1).pdf(support), 
               'k', alpha=0.1)  


In [ ]:
# t_df=2 samples have larger magnitude
(np.abs(y)>np.abs(x)).mean()-0.5

In [ ]:
# fraction of standarized normal > 3
(np.abs(x)>3).mean()

In [ ]:
import pymc as pm

# 创建 PyMC 模型
with pm.Model() as spike_and_slab:
    
    # 定义一个 Beta 分布的先验概率 p，alpha 和 beta 都为 1 (均匀分布)
    p = pm.Beta('p', alpha=1, beta=1)
    
    # 定义 spike 部分，使用 Bernoulli 分布，参数为 p，shape=t 表示生成 t 个样本
    spike = pm.Bernoulli('spike', p=p, shape=t)
    
    # 定义 slab 部分，使用 Normal 分布，均值为 0，标准差为 100，生成 t 个样本
    slab = pm.Normal('slab', mu=0, sigma=100, shape=t)
    
    # 定义观测数据 test，服从 Normal 分布
    # 均值为 spike 与 slab 的乘积，标准差为 1
    # observed=y 表示我们使用观测数据 y 来对模型进行拟合
    test = pm.Normal('test', mu=spike * slab, sigma=1, observed=y)
    
    # 对模型进行采样，返回采样结果
    idata = pm.sample()


In [ ]:
import arviz as az

az.plot_trace(idata, var_names='p');

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(10,3))

c,d = idata.posterior['slab'].values.shape[:2]

outlier = (np.abs(y)>3)
col = ['r' if o else 'k' for o in outlier]
ax[0].plot([y.min(),y.max()], [y.min(),y.max()])
ax[0].scatter(y,
              (idata.posterior['slab'].values.reshape(c*d,t)*
               idata.posterior['spike'].values.reshape(c*d,t)).mean(axis=0),
              color=col, alpha=0.2)

ax[-1].plot(y,
            idata.posterior['spike'].values.reshape(c*d,t).mean(axis=0),'k.');

In [ ]:
with pm.Model() as horseshoe:    
    
    # 定义 tau_0 的值为 0.1，作为全局尺度参数的标准差
    # tau_0 也可以定义为一个随机变量，例如：pm.Beta('p_non0', alpha=1, beta=1)
    tau_0 = 0.1  
    
    # 定义全局尺度参数 tau，使用 Half-Cauchy 分布
    tau = pm.HalfCauchy('tau', beta=tau_0, shape=1)
    
    # 定义局部尺度参数 lambdas，使用 Half-Cauchy 分布
    lambdas = pm.HalfCauchy('lambdas', beta=1, shape=t)
    
    # 定义一个标准正态分布的参数 mu_，用来生成 mu 的初始值
    mu_ = pm.Normal('mu_', mu=0, sigma=1, shape=t)
    
    # 将 mu_ 通过 tau 和 lambdas 进行缩放，生成实际使用的 mu
    mu = pm.Deterministic('mu', mu_ * tau * lambdas)
    
    # 定义观测变量 test，假定其服从均值为 mu、标准差为 1 的正态分布
    # observed=y 表示将观测数据 y 用于模型拟合
    test = pm.Normal('test', mu=mu, sigma=1, observed=y)
    
    # 对模型进行采样，并保存采样结果
    idata2 = pm.sample()  
 

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(10,3))

outlier = (np.abs(y)>3)
col = ['r' if o else 'k' for o in outlier]
ax[0].plot([y.min(),y.max()], [y.min(),y.max()])
ax[0].scatter(y,
              idata2.posterior['mu'].values.reshape(c*d,t).mean(axis=0),
              color=col, alpha=0.2)

ax[1].hist(1/(1+(idata2.posterior['tau'].values*
                 idata2.posterior['lambdas'].values).reshape(c*d,t).mean(axis=0)**2));

ax[2].scatter(idata2.posterior['mu'].values.reshape(c*d,t).mean(axis=0),
              (idata.posterior['slab'].values.reshape(c*d,t)*
               idata.posterior['spike'].values.reshape(c*d,t)).mean(axis=0),
              color=col, alpha=0.2)
ax[2].set_ylabel("Spike and Slab")
ax[2].set_xlabel("Horseshoe")

plt.tight_layout()

## Bayesian Occam's Razor

$\begin{align*}
\Pr(M|x) &\propto{} \int f(x|\theta,M) p(\theta|M) p(M) d \theta\\
\Pr(M[u_i=0]|y_i) &\propto{} \int N(y_i|0,1) (1-p)^{1-u_i} p(p) dp = \int_0^1 N(y_i|0,1) (1-p)^1 dp = N(y_i|0,1)\\
\Pr(M[u_i=1]|y_i) &\propto{} \int\int N(y_i|\theta,1) N(\theta|0,100) p^{u_i} p(p) dp d\theta = \int N(y_i|\theta,1) N(\theta|0,100) d\theta \\
\end{align*}$

In [ ]:
from scipy import integrate

def Pr_M_u_0(y):
    return stats.norm(0,1).pdf(y)

@np.vectorize
def Pr_M_u_1(y):
    return integrate.quad(lambda 𝜃: stats.norm(𝜃,1).pdf(y)*stats.norm(0,100).pdf(𝜃), 
                          -500, 500)[0]

fig,ax = plt.subplots(1,2,figsize=(10,3))

support = np.linspace(-10,10,100)
Pr_M_u_0_ = Pr_M_u_0(support)
Pr_M_u_1_ = Pr_M_u_1(support)

ax[0].plot(support, Pr_M_u_0_, label="$\\propto \\Pr(M[u_i=0]|y_i)$")
ax[0].plot(support, Pr_M_u_1_, label="$\\propto \\Pr(M[u_i=1]|y_i)$")
ax[0].legend()

ax[1].plot(support, Pr_M_u_1_/(Pr_M_u_1_+Pr_M_u_0_), label="$\\Pr(M[u_i=1]|y_i)$")
ax[1].legend();


The **Bayesian Occam's razor** phenomenon is a **curse of (parameter) dimensionality** which encourages simpler models to have larger marginal likelihoods than more complex models. We can generally expect that

$$p(\mathbf{x}|M_{\theta}) = \int f(\mathbf{x}|\theta) p(\theta)d\theta > \int\int f(\mathbf{x}|\theta, \eta) p(\theta, \eta) d\theta d\eta = p(\mathbf{x}|M_{\theta, \eta})$$

because the density of the prior $p(\theta, \eta)$ must be reduced to in relation to the increased volume in increasingly higher demensional (parameter) space.

---

Marginal likelihoods are small if
- prior specification are badly non-overlapping with the likelihood
- or if the data model cannot adequately reflect the empirical data distribution

Added model complexity that sufficiently increases likelihood to overcome the **Bayesian Occam's razor** penalization on the marginal likelihood would seem to be justified, but what of decreasing marginal likelihoods with increasing model complexity?

- The **Bayesian Occam's razor** is not a model selection tool because it depends on how well-aligned the prior is to the likelihood (data)
- What the **Bayesian Occam's razor** shows is that priors have increasingly influential effects in higher dimesions, and prior-data conflict in increasingly higher dimensions is increasingly problematic

- Bayesian Occam's razor 是一种 curse of dimensionality 现象，当模型复杂度增加时，先验分布的密度必须在更大的参数空间中扩展，从而降低了 marginal likelihood。

- 如果先验分布与 likelihood 的重叠较小或数据模型无法准确反映经验数据分布，则 marginal likelihood 会很小。

- 在更高维度中，Bayesian Occam's razor 强调了先验的影响力增加，因此 prior-data conflict 在更高维度下会变得更加严重。


## Bayes Factors

$K = \frac{p(\mathbf{x}|M_1)}{p(\mathbf{x}|M_0)} = \frac{p(M_1|\mathbf{x})}{p(M_0|\mathbf{x})} \frac{p(M_0)}{p(M_1)}$ is often proposed as a model comparison tool with evidence in favor of $M_1$ over $M_0$ given as 

|log10 K|	K|	Strength of evidence |
|-|-|-|
|0 to 1/2	|1 to 3.2	| Anecdotal|
|1/2 to 1	|3.2 to 10	| Substantial |
|1 to 2	|10 to 100 |	Strong|
|> 2|	> 100	|Decisive|
    
For fair "apples to apples" comparisons Bayes Factors might be reasonably useful.

For example, the **Savage-Dickey density ratio** for the special case where $M_1$ is a  (nested model) version of $M_0$ where the parameter value $\theta = \theta_1$ is the ratio of the **posterior** and the **prior** evaluated at $\theta=\theta_1$ 

$$
\begin{align*}
p(\mathbf{x}|M_1) ={}& \int p(\mathbf{x},\eta | M_1)d\eta\\
 ={}& \int p(\mathbf{x},\eta|\overbrace{\theta=\theta_1, M_0}^{=M_1}) d\eta = p(\mathbf{x}|\theta=\theta_1, M_0)\\
\frac{p(\mathbf{x}|M_1)}{p(\mathbf{x}|M_0)} ={}& \frac{p(\mathbf{x}|\theta=\theta_1, M_0)}{p(\mathbf{x}|M_0)} = \frac{\frac{p(\theta=\theta_1|\mathbf{x}, M_0) p(\mathbf{x}| M_0)}{p(\theta=\theta_1|M_0)}}{p(\mathbf{x}|M_0)}\\ ={}& \frac{p(\theta=\theta_1|\mathbf{x}, M_0)}{p(\theta=\theta_1|M_0)}
\end{align*}$$

Bayes Factors 是一种模型比较工具，通过计算模型 M1 和 M0 在数据下的边际似然比来量化数据对模型支持的强度，其 log10 K 值可以被划分为 Anecdotal、Substantial、Strong 和 Decisive 等等级。
在嵌套模型中，Savage-Dickey density ratio 用于直观地表达 Bayes Factor，即通过比较特定参数值处的 posterior 与 prior，展示数据如何改变我们对该参数的信念。

## Bayesian Information Sharing

Returning to our initial variable selection considerations...

在变量选择中，Bayesian Information Sharing 指的是利用层次模型或共用先验，使得不同变量能够共享信息，从而提高弱信号的识别能力和参数估计的稳定性。这种信息共享方法可以通过 spike‐and‐slab、horseshoe 等模型实现，有助于在高维数据中有效抑制噪声变量的影响并改善预测性能。








In [ ]:
fig,ax = plt.subplots(1,2,figsize=(10,3))

bounds = np.abs(y)<10

ax[0].plot(support, (Pr_M_u_1_/(Pr_M_u_1_+Pr_M_u_0_)), label="$\\Pr(M[u_i=0]|y_i)$")
ax[0].plot(y[bounds],
           idata.posterior['spike'].values.reshape(c*d,t).mean(axis=0)[bounds],'k.');
ax[0].set_title("Theory for a single observation doesn't match...")

ax[1].plot(support, (.08*Pr_M_u_1_/(.08*Pr_M_u_1_+.92*Pr_M_u_0_)), label="$\\Pr(M[u_i=0]|y_i)$")
ax[1].plot(y[bounds],
           idata.posterior['spike'].values.reshape(c*d,t).mean(axis=0)[bounds],'k.');
ax[1].set_title("Because the observations influence each other...");


### Mixture Models 

A very simple form of information sharing...

$$\scriptsize
\begin{align*}
x_i \sim {} & \sum_{k=1}^K \mathbf{v}_{ik} \mathcal N (\mu_k,\sigma_k^2) & \mu_k \sim {} & \mathcal N (\mu_{k0},\sigma_{k0}^2) \quad \sigma_k^2 \sim \text{Inverse-Gamma} (\alpha_{k0}, \beta_{k0})\\
\overset{\overset{\text{multinomial}}{\mathbf{v}_i\,\sim\,\text{MN}}(\mathbf{p}, \,n=1)}{\Pr(\mathbf{v}_i|E[\mathbf{v}_i]=\mathbf{p}, n=1)} = {}& \frac{n!}{v_1!\cdots v_K!} p_1^{\mathbf{v}_{i1}} \cdots p_K^{\mathbf{v}_{iK}}  & \sum_{j=1}^n \mathbf{v}_{ik} = {}& 1 \quad \mathbf{v}_{ik} \in \{0,1\} \quad \text{latent (unknown) subpulation membership $\textbf{v}$} \\
\underset{\text{Dirichlet}}{\overset{p\,\sim\,\text{Dir}(\boldsymbol \alpha)}{p\left(\mathbf{p}|\boldsymbol \alpha \right)}} = {}& {\frac {1}{\mathrm {B} ({\boldsymbol {\alpha }})}}\prod _{k=1}^{K}p_{k}^{\alpha _{k}-1} & \sum_{j=1}^np_k = {}& 1 \quad {\displaystyle \mathrm {B} ({\boldsymbol {\alpha }})= \prod \limits _{k=1}^{K}\Gamma (\alpha _{k}) \bigg/ \Gamma \left(\sum \limits _{k=1}^{K}\alpha _{k}\right)} \quad E[p_k] = \alpha_k\bigg/\sum_{k=1}^K \alpha_k 
\end{align*}$$

$$\scriptsize
\begin{align*}
p(\mu_k | -) \propto {} & \mathcal N (\mu_k| \mu_{k0},\sigma_{k0}^2) \prod_{i=1}^n \sum_{k=1}^K\mathbf{v}_{ik} \mathcal N (x_i | \mu_k,\sigma_k^2) & p(\sigma_k^2 | -) \propto {} & \underset{\text{Inverse-Gamma}}{\text{IG} (\sigma_k^2|\alpha_{k0}, \beta_{k0})} \prod_{i=1}^n \sum_{k=1}^K\mathbf{v}_{ik} \mathcal N (x_i|\mu_k,\sigma_k^2)
\end{align*}$$$$\scriptsize
\begin{align*}
\Pr(\mathbf{v}_{ik}=1|-) \propto {} & p_1^{\mathbf{v}_{i1}} \cdots p_K^{\mathbf{v}_{iK}} \sum_{k=1}^K\mathbf{v}_{ik} \mathcal N (x_i | \mu_k,\sigma_k^2) & p(\mathbf{p}| - ) \propto {}& \prod _{k=1}^{K}p_{k}^{\alpha _{k}-1} \prod_{i=1}^{n} p_1^{\mathbf{v}_{i1}} \cdots p_K^{\mathbf{v}_{iK}}
\end{align*}$$

在这种简单的信息共享模型中，每个观测值 $x_i$ 被假定来自于 $K$ 个子群体的混合正态分布，每个子群体有各自的参数 $\mu_k$ 和 $\sigma_k^2$。每个观测的子群体归属由潜在指示变量 $\mathbf{v}_i$ 表示，其先验服从多项分布（multinomial），而混合权重 $\mathbf{p}$ 则服从 Dirichlet 分布。

这种结构使得各子群体之间的信息可以共享：在更新 $\mu_k$、$\sigma_k^2$、$\mathbf{v}$ 以及 $\mathbf{p}$ 的后验分布时，所有观测数据均参与联合推断，从而提高了参数估计的稳定性和准确性。


In [ ]:
np.random.seed(9)

k = 5  # 子分布的数量
alpha = [2] * k  # Dirichlet 分布的参数，用于生成 p_true
p_true = stats.dirichlet(alpha).rvs(1)[0]  # 从 Dirichlet 分布中生成权重 (p_true)
# p_true.sum() 确认 p_true 的总和为 1

# 从标准正态分布（均值为 0，标准差为 3）中生成 k 个均值
mu_k_true = stats.norm(0, 3).rvs(k)

# 定义支持区间 support，从 -6 到 9，包含 100 个点
support = np.linspace(-6, 9, 100)

# 初始化 population_pdf 为零向量，用于累积所有子分布的 PDF
population_pdf = 0 * support

# 从 Half-Normal 分布中生成 k 个标准差的平方 (sigma2_k_true)
sigma2_k_true = stats.halfnorm().rvs(k)

fig, ax = plt.subplots(1, 2, figsize=(18, 5))

ax[0].bar(x=np.linspace(1, 5, 5), height=p_true)

for j in range(k):
    # 计算第 j 个子分布的 PDF
    subpopulation_pdf = p_true[j] * stats.norm(mu_k_true[j], sigma2_k_true[j] ** 0.5).pdf(support)
    # 绘制第 j 个子分布的 PDF 曲线
    ax[1].plot(support, subpopulation_pdf)
    # 将当前子分布的 PDF 累加到 population_pdf
    population_pdf += subpopulation_pdf

# 绘制累加后的总人口分布 (population_pdf)
ax[1].plot(support, population_pdf)


In [ ]:
n_ = 1000
v_true = stats.multinomial(n=1,p=p_true).rvs(n_)
v_true[:3,:]

In [ ]:
print(mu_k_true)
print((v_true*mu_k_true)[:3,:])
print((v_true*mu_k_true).sum(axis=1)[:3])

In [ ]:
# 根据权重 v_true 和每个子分布的参数，生成合成数据 x_
x_ = stats.norm(
    (v_true * mu_k_true).sum(axis=1),                  # 均值为 v_true 与 mu_k_true 的加权平均
    (v_true * sigma2_k_true).sum(axis=1) ** 0.5        # 标准差为 v_true 与 sigma2_k_true 加权总和的平方根
).rvs()  # 从上述正态分布中抽样

# 在第一个子图中绘制生成的数据的直方图
ax[0].hist(x_)

# 显示图形
fig

### Hierachical (more than random effects) Models

A more interesting form of information sharing...


In [ ]:
n = 100  # 总共生成 n 个均值
mu = np.sort(stats.norm(0, 1).rvs(n))  # 从标准正态分布中生成 n 个均值，并进行排序

r = 5  # 每个 mu 对应生成 r 个相关的观测值

# 创建相关的观测数据 y
# 对于每一个 mu，生成 r 个正态分布的样本，均值为 mu，标准差为 1
y = stats.norm(mu, 1).rvs((r, n))

# 将 y 转换为整数，得到 ndx
ndx = y.astype(int)

# 将 ndx 中的每一列都重写为列索引值 (0 到 99)
for j in range(100):
    ndx[:, j] = j

# 计算 y 的相关矩阵
np.round(np.corrcoef(y), 2)


In [ ]:
# 去相关化通过组内估计 (Decorrelation by within group estimation)
# 使用独立的先验分布 (independent priors)

with pm.Model() as MixEff:
    
    # 定义 m 的先验分布：每个均值 m 独立地服从 N(0, 10) 分布
    m = pm.Normal('m', mu=0, sigma=10, shape=n)
    
    # 定义 s 的先验分布：每个标准差 s 独立地服从 Half-Normal 分布，标准差为 10
    s = pm.HalfNormal('s', sigma=10, shape=n)
    
    # 定义观测数据 x，使用给定的 ndx 和 y 进行分组估计
    # 这里的 m 和 s 是根据 ndx 中的索引值来选取的
    x = pm.Normal(
        'x', 
        mu=m[ndx.flatten()],     # 通过 ndx.flatten() 对应的索引从 m 中提取相应的均值
        sigma=s[ndx.flatten()],   # 同理，从 s 中提取相应的标准差
        observed=y.flatten()      # 展平后的观测数据作为输入
    )
    
    # 对模型进行采样，返回采样结果
    idata = pm.sample()

In [ ]:
c,d = idata.posterior.m.shape[:2]
plt.plot(mu, idata.posterior.m.values.reshape(c*d,n).mean(axis=0), '.')
plt.plot([-3,3],[-3,3]);


In [ ]:
az.plot_trace(idata, var_names='s');

In [ ]:
az.plot_trace(idata, var_names='m');

In [ ]:
# 去相关化通过组内估计 (Decorrelation by within group estimation)
# 使用公共的分层先验分布 (common hierarchical priors)

with pm.Model() as MixEff2:
    
    # 对 m 的先验分布使用分层结构 (hierarchical structure)
    # 定义 m_s 为 Half-Normal 分布，控制 m 的标准差的整体尺度
    m_s = pm.HalfNormal('m_s', sigma=10)
    
    # 定义全局的均值 m_，服从 Normal 分布，标准差为 m_s
    m_ = pm.Normal('m_', mu=0, sigma=m_s)
    
    # 定义每个组内的 m，服从 Normal 分布，以 m_ 和 m_s 作为其均值与标准差
    m = pm.Normal('m', mu=m_, sigma=m_s, shape=n)
    
    # 对 s 的先验分布也使用分层结构
    # 定义 s_s 和 s_，分别服从 Half-Normal 分布
    s_s = pm.HalfNormal('alpha', sigma=10)
    s_ = pm.HalfNormal('beta', sigma=10)
    
    # 定义每个组内的 s，服从 Gamma 分布，使用 s_s 和 s_ 作为其 alpha 和 beta 参数
    s = pm.Gamma('s', alpha=s_s, beta=s_, shape=n)
    
    # 定义观测数据 x，使用给定的 ndx 和 y 进行分组估计
    # mu 和 sigma 是从分层先验中提取的
    x = pm.Normal(
        'x', 
        mu=m[ndx.flatten()],             # 从 m 中提取相应的均值
        sigma=s[ndx.flatten()],          # 从 s 中提取相应的标准差
        observed=y.flatten()             # 展平后的观测数据作为输入
    )
    
    # 对模型进行采样，返回采样结果
    idata2 = pm.sample()


In [ ]:
az.plot_trace(idata2, var_names=['alpha','beta','s'])
plt.tight_layout()
print(y.std(axis=0).mean(),y.std(axis=0).std())

In [ ]:
az.plot_trace(idata2, var_names=['m_','m_s','m'])
plt.tight_layout()
print(y.mean(axis=0).mean(),y.mean(axis=0).std())

In [ ]:
c,d = idata2.posterior.m.shape[:2]
plt.plot(mu, idata.posterior.m.values.reshape(c*d,n).mean(axis=0),
         '.', label="Independent Inference:\nNo Information Sharing")
plt.plot(mu, idata2.posterior.m.values.reshape(c*d,n).mean(axis=0),
         '.', label="Inference on shared priors:\nInformation Sharing")
plt.plot([-3,3],[-3,3])
plt.legend();

In [ ]:
((mu-idata.posterior.m.values.reshape(c*d,n).mean(axis=0))**2).sum()**0.5         

In [ ]:
((mu-idata2.posterior.m.values.reshape(c*d,n).mean(axis=0))**2).sum()**0.5

In [ ]:
# 这里是一个更简单的分层模型 (Simplified hierarchical model)

with pm.Model() as MixEff3:
    
    # 定义 m 的先验分布，假定每个 m 独立地服从标准正态分布 N(0, 1)
    m = pm.Normal('m', mu=0, sigma=1, shape=n)
    
    # 定义 s 的先验分布，使用 Gamma 分布进行建模
    # alpha 和 beta 都设置为 16，这使得分布更加集中
    s = pm.Gamma('s', alpha=16, beta=16, shape=n)
    
    # 定义观测数据 x，使用给定的 ndx 和 y 进行分组估计
    x = pm.Normal(
        'x', 
        mu=m[ndx.flatten()],        # 从 m 中提取相应的均值
        sigma=s[ndx.flatten()],     # 从 s 中提取相应的标准差
        observed=y.flatten()        # 展平后的观测数据作为输入
    )
    
    # 对模型进行采样，并保存采样结果
    idata3 = pm.sample()


In [ ]:
c,d = idata.posterior.m.shape[:2]
plt.plot(mu, idata3.posterior.m.values.reshape(c*d,n).mean(axis=0), '.')
plt.plot([-3,3],[-3,3]);


In [ ]:
((mu-idata3.posterior.m.values.reshape(c*d,n).mean(axis=0))**2).sum()**0.5         

In [ ]:
# 一种 hacky 的方法展示更多的信息共享 (Something hacky just to show more information sharing)

with pm.Model() as MixEff4:
    
    # 定义 m 的先验分布：每个 m 独立地服从标准正态分布 N(0, 1)
    # 这里的 m 的维度为 n+4，允许后续索引时超出原有范围
    m = pm.Normal('m', mu=0, sigma=1, shape=n+4)
    
    # 定义 s 的先验分布：每个 s 独立地服从 Gamma 分布
    # 使用 alpha=16, beta=16 来保持与之前模型的一致性
    s = pm.Gamma('s', alpha=16, beta=16, shape=n+4)
    
    # 定义观测数据 x
    # 在这里，我们引入了一种“滑动窗口”平均的方法来共享信息
    x = pm.Normal(
        'x', 
        mu=(m[ndx.flatten()] + m[ndx.flatten() + 1] + m[ndx.flatten() + 2] 
            + m[ndx.flatten() + 3] + m[ndx.flatten() + 4]) / 5,  # 将相邻的 5 个 m 的平均作为均值
        sigma=(s[ndx.flatten()] + s[ndx.flatten() + 1] + s[ndx.flatten() + 2] 
               + s[ndx.flatten() + 3] + s[ndx.flatten() + 4]) / 5,  # 将相邻的 5 个 s 的平均作为标准差
        observed=y.flatten()  # 展平后的观测数据作为输入
    )
    
    # 对模型进行采样，并保存采样结果
    idata4 = pm.sample()


In [ ]:
az.plot_trace(idata4, var_names=['m','s'])
plt.tight_layout()
print(y.mean(axis=0).mean(),y.mean(axis=0).std())

In [ ]:
idata2.posterior['m'].values.reshape(c*d,n).std(axis=0)

In [ ]:
((idata4.posterior['m'].values[:,:,:-4]+
  idata4.posterior['m'].values[:,:,1:-3]+
  idata4.posterior['m'].values[:,:,2:-2]+
  idata4.posterior['m'].values[:,:,3:-1]+
  idata4.posterior['m'].values[:,:,4:])/5).reshape(c*d,n).std(axis=0)

In [ ]:
mmm = \
((idata4.posterior['m'].values[:,:,:-4]+
  idata4.posterior['m'].values[:,:,1:-3]+
  idata4.posterior['m'].values[:,:,2:-2]+
  idata4.posterior['m'].values[:,:,3:-1]+
  idata4.posterior['m'].values[:,:,4:])/5).reshape(c*d,n).mean(axis=0)

In [ ]:
c,d = idata.posterior.m.shape[:2]
plt.plot(mu, mmm, '.')
plt.plot([-3,3],[-3,3]);


In [ ]:
((mu-mmm)**2).sum()**0.5

In [ ]:
_lambda = .06  # 控制相似性矩阵的平滑度 (smoothing scale)
sigma2 = 1     # 控制相似性矩阵的幅度 (scaling factor)

# 计算每个样本的均值 (按列计算)
y_mean = y.mean(axis=0)

# 创建相似性矩阵 K，使用高斯核 (Gaussian kernel)
K = sigma2 * np.exp(
    - (y_mean.reshape(n, 1) - y_mean.reshape(1, n)) ** 2 / _lambda ** 2
)

# 修改相似性矩阵 K，使用绝对差值形式 (Manhattan distance) 计算
alf = 1  # 控制相似性矩阵中衰减的速率
K = sigma2 * np.exp(
    - alf * np.abs(y_mean.reshape(n, 1) - y_mean.reshape(1, n))
)

# 对称化相似性矩阵 K，使得 K 矩阵为对称矩阵
K = (K + K.T) / 2

# 显示相似性矩阵 K 的图像
plt.imshow(K)
plt.colorbar()

In [ ]:
K = sigma2*np.exp(-alf*np.abs(mu.reshape(n,1) - 
                              mu.reshape(1,n)))
K=(K+K.T)/2  # np.linalg.matrix_rank(K)
plt.imshow(K)
plt.colorbar();

In [ ]:
# What we've learned
with pm.Model() as MixEff5:
    
    # 定义 m 的先验分布为多元正态分布 (Multivariate Normal Distribution)
    # 使用之前计算的相似性矩阵 K 作为协方差矩阵 (cov)
    # initval 指定初始值为 y 的均值 (按列计算)
    m = pm.MvNormal('m', mu=np.zeros(n), cov=K, initval=y.mean(axis=0))
    
    # 定义 s 的先验分布，使用 Gamma 分布
    # alpha 和 beta 都设置为 16，保持一致性
    s = pm.Gamma('s', alpha=16, beta=16, shape=n)
    
    # 定义观测数据 x，使用指定的 m 和 s 对应的均值与标准差
    x = pm.Normal(
        'x', 
        mu=m[ndx.flatten()],          # 从 m 中提取相应的均值
        sigma=s[ndx.flatten()],       # 从 s 中提取相应的标准差
        observed=y.flatten()          # 展平后的观测数据作为输入
    )
    
    # 对模型进行采样，并保存采样结果
    idata5 = pm.sample()


In [ ]:
c,d = idata.posterior.m.shape[:2]
plt.plot(mu, idata5.posterior.m.values.reshape(c*d,n).mean(axis=0), '.')
plt.plot([-3,3],[-3,3]);


In [ ]:
az.plot_trace(idata5, var_names=['m','s'])
plt.tight_layout()
print(y.mean(axis=0).mean(),y.mean(axis=0).std())

In [ ]:
((mu-idata5.posterior.m.values.reshape(c*d,n).mean(axis=0))**2).sum()**0.5         

In [ ]:
plt.plot(mu, label="truth")
plt.plot(y.mean(axis=0), label="observed")
plt.plot(np.sort(y.mean(axis=0)), label="rank")
plt.legend();

In [ ]:
K = sigma2*np.exp(-alf*np.abs(np.sort(y).mean(axis=0).reshape(n,1) - 
                              np.sort(y).mean(axis=0).reshape(1,n)))

K=(K+K.T)/2  # np.linalg.matrix_rank(K)
plt.imshow(K)
plt.colorbar();

In [ ]:
# What we've learned
with pm.Model() as MixEff6:
    
    # 定义 m 的先验分布为多元正态分布 (Multivariate Normal Distribution)
    # 使用之前计算的相似性矩阵 K 作为协方差矩阵 (cov)
    # initval 指定初始值为 y 的列均值 (按列计算)
    m = pm.MvNormal('m', mu=np.zeros(n), cov=K, initval=y.mean(axis=0))
    
    # 定义 s 的先验分布，使用 Gamma 分布
    # alpha 和 beta 都设置为 16，确保分布相对集中
    s = pm.Gamma('s', alpha=16, beta=16, shape=n)
    
    # 定义观测数据 x，使用重新排序后的 y 作为观测值
    # np.argsort(y.mean(axis=0)) 对 y 的列均值进行排序，并对 y 进行相应的列重排
    x = pm.Normal(
        'x', 
        mu=m[ndx.flatten()],                # 从 m 中提取相应的均值
        sigma=s[ndx.flatten()],             # 从 s 中提取相应的标准差
        observed=y[:, np.argsort(y.mean(axis=0))].flatten()  # 对 y 进行排序后展平
    )
    
    # 对模型进行采样，并保存采样结果
    idata6 = pm.sample()


In [ ]:
c,d = idata.posterior.m.shape[:2]
plt.plot(mu, idata6.posterior.m.values.reshape(c*d,n).mean(axis=0), '.')
plt.plot([-3,3],[-3,3]);


In [ ]:
((mu-idata6.posterior.m.values.reshape(c*d,n).mean(axis=0))**2).sum()**0.5         

## Back to Model Selection

### LRT

The **(nested) log likelihood ratio test (LRT) statistic** is $\quad2\log\left(\frac{p(y|\hat \theta_{M_1})}{p(y|\hat \theta_{M_0)}}\right) = \color{gray}{2(\log(p(y|\hat \theta_{M_1})) - \log(p(y|\hat \theta_{M_0})))} $ 

and has an asymptotically $\;\chi^2_{df}\;$ distribution with **expected value** [equal](https://en.wikipedia.org/wiki/Likelihood-ratio_test) **degrees of freedom**  $\dim(\theta_{M_1})-\dim(\theta_{M_0})$

$$2\log\left(\frac{p(y|X_{n\times p}\hat \beta)}{p(y|\bar y)}\right)\quad \text{is asymptotically} \;\;\chi^2_{p-1}\;\;\text{and a large LRT statistic rejects $M_0$ in favor of $M_1$} $$

> #### Sketch of Some "Intuition" as to Why this is so
> 
> - $p(y|\hat \theta_{M})$ is assymptotically normal 
> - $\frac{p(y|\hat \theta_{M_1})}{p(y|\hat \theta_{M_0})}$ cancels normalizing constants
> - $\log p(y|\hat \theta_{M})$ after canceling normalizing constants, and since one parameter $\theta_i$ perfectly predicts one $\theta_i=y_i$, is $-\frac{1}{2}\sum_{i=1}^{n-\text{dim}(M)+1} \frac{(y_i-\bar y)^2}{\sigma^2} -\frac{1}{2}\sum_{i=1}^{\text{dim}(M)-1} \frac{(y_j-y_j)^2}{\sigma^2}$
> - $E[\frac{(y_i-\bar y)^2}{\sigma^2}] = 1$ so the $-\frac{1}{2}$ term makes difference $\text{dim}(M)-1$ and cancels the scaling $2$

(nested) log likelihood ratio test (LRT) statistic 用于比较两个嵌套模型的适配性，其统计量为 $2\log\left(\frac{p(y|\hat \theta_{M_1})}{p(y|\hat \theta_{M_0})}\right)$，渐进分布为 $\chi^2_{df}$，期望值等于自由度差 $\dim(\theta_{M_1}) - \dim(\theta_{M_0})$。

基本直觉是：$p(y|\hat \theta_{M})$ 渐进服从正态分布，$\frac{p(y|\hat \theta_{M_1})}{p(y|\hat \theta_{M_0})}$ 消去标准化常数，$\log p(y|\hat \theta_{M})$ 的期望值中，每一个参数对数据的完美预测贡献值为 $-1/2$，差异的期望值为 $\text{dim}(M)-1$ 并消去缩放因子 2。

### Deviance  and "Bayesian" Model Size

Model **deviance** is $\quad D(\theta) = 2(\underset{\text{a constant}}{\overset{\text{saturated model}}{\log(p(y|y))}} - \overset{\text{actual model}}{\log(p(y|\theta)})) \color{gray}{= 2\log\left(\frac{p(y|y)}{p(y|\theta)}\right)} \geq 0 \quad$ (scaled by $2$ [to match the LR](https://stats.stackexchange.com/qu,estions/379810/why-is-the-deviance-defined-with-a-factor-2-or-likelihood-ratio-squared))

The [Bayesian characterization](https://citeseerx.ist.psu.edu/document?repid=rep1&type=pdf&doi=d78ad2497639bff740d0c1181c35263d2630b172)
 of the **effective number of parameters** in a model is

$$p_D = {\overline{D(\theta)} - D(\bar{\theta})} \color{gray}{= 2(\log(p(y|\bar{\theta})) - \overline{\log(p(y|\theta))}) \underset{\text{usually}}{\geq 0}} \quad \text{ or } \quad \underset{\text{not typically preferred as it's less stable}}{p_D = 2\text{Var}_{p(\theta|y)}[\log(p(y|\theta))]}$$

because (for fixed models such as ["linear models with uniform prior distributions"](http://www.stat.columbia.edu/~gelman/research/published/waic_understand3.pdf) and large $n$) both versions of $p_D$ estimate the degrees of freedom parameter of the asymptotic $\chi^2_{\text{df}}$ of $\log(p(y|\theta))$ which is the number of parameters $p(y|\theta)$.

> #### Sketch of the "Proof" as to Why this is so
> 
> $-2\log(p(y|\bar{\theta})$ for linear regression is the expected standardized residual sum of squares
> 
> $\begin{align*}
(y-\hat y)^T(y-\hat y)/\sigma^2 &={} y^T(I-H)^T(I-H)y/\sigma^2
= y^T(I-H)(I-H)y/\sigma^2 
= y^T(I-H)y/\sigma^2 \\
&={} (X\beta + \epsilon)^T(I-H)(X\beta + \epsilon)/\sigma^2
= \epsilon^T(I-H)\epsilon/\sigma^2 = \text{trace}(\epsilon^T(I-H)\epsilon/\sigma^2)\\
&={} \text{trace}((I-H)\epsilon\epsilon^T/\sigma^2) = \text{trace}((I-H)\epsilon\epsilon^T/\sigma^2) \\
\text{with expected value} & \quad \; \text{trace}((I-H)\sigma^2I/\sigma^2) = \text{trace}(I-H) = n - \text{trace}(H)\\
&={} n - \text{trace}(X^T(X^TX)^{-1}X) = n - \text{trace}((X^TX)^{-1}XX^T) = n - \text{trace}(I_{p \times p})\\
&={} n - p
\end{align*}$
>
> while $-2\overline{\log(p(y|\theta))}$ does something like reflect the variability of $n-p$ unexplained data points plus the variability of $p$ parameters 
> $E_{\hat y}[\sum_{i=1}^n \frac{(y_i - \hat y_i)^2}{\sigma^2} ] \overset{\text{if unbiased}}{\approx} E_{\hat y}[\sum_{i=1}^n \frac{(y_i - E[y_i])^2}{\sigma^2} ]$ which has an expected value (with respect to $y$) of $n$.

模型deviance 用于衡量模型对数据的拟合程度，通过与“完美模型”的似然比较来计算，值越小说明模型拟合越好。有效参数数量 (p_D) 用于描述模型的复杂性，主要通过两种方法估计：一是比较参数均值的deviance与所有样本的平均deviance的差异，二是计算log-likelihood在后验分布中的变异性，复杂模型的p_D会更大，因为其拟合依赖于精确的参数而非参数的平均值。

### Information Criterion

**Information Criterion** criterion refer to the fact that each additional parameter is expected to decrease $-2\log f(y | \theta)$ by $1$ unit, where a lower negative loglikelihood is "better". 

Rearranging $\;\;p_D = {\overline{D(\theta)} - D(\bar{\theta})}\;\;$ in terms of **posterior mean deviance** $\;\;\overline{D(\theta)} = D(\bar{\theta}) + p_D\;\;$ leads to the ["adequacy" "measure of fit plus complexity"](https://fisher.stats.uwo.ca/faculty/aim/2015/9938/articles/SpiegelhalterJRSSB2002.pdf) **deviance information criterion** in the standard **information criterion form**

\begin{align*}
\text{DIC:} \;{}& -2\log(p(y|\bar{\theta})) + 2p_D \quad\;\;\text{since} \quad \overline{D(\theta)}+p_D = D(\bar{\theta}) + 2p_D \quad \text{$p(y|y)$'s cancel in DIC$_{M_1}$-DIC$_{M_0}$}\\
\text{AIC:} \;{}& -2\log(p(y|\hat{\theta})) + 2p \quad\quad\, \text{and $\quad e^{(\text{AIC$_{M_0}$-AIC$_{M_1}$})/2}\quad$ is an unnested version of the }\textbf{LRT statistic}\\
\text{BIC:} \;{}& -2\log(p(y|\hat{\theta})) + p\ln(n) \;\;\, \text{which approximates }\textbf{Bayes Factor} K = \frac{p(\mathbf{x}|M_1)}{p(\mathbf{x}|M_0)} \approx e^{(BIC_{M_0}-BIC_{M_1})/2}
\end{align*}

### Out of sample predictive performance 

Recall that **Bayes factors** induce an **Occam's razor** (parameter integration dimension) penalization for model complexity; whereas, **DIC** and [**AIC**](https://stats.stackexchange.com/questions/116935/comparing-non-nested-models-with-aic) measure ["out-of-sample-prediction error using a bias-corrected adjustment of within-sample error"](http://www.stat.columbia.edu/~gelman/research/published/waic_understand3.pdf) 

Even though the **Bayesian Occam's razor** (with **Bayes factors**) naturally favoring parsimony in model selection is intuitively attractive, this "simplest solution is the best solution" perspective is not necessarily always justified.

- It is still reasonable to prefer more complex models with improved out of sample performance 

**Machine learning $K$-folds cross-validation parameter tuning** optimizes the *bias-variance tradeoff** in a model fit which allows the model to be both underfit and overfit in different areas of the prediction space and so improves the overall out of sample predictive accuracy by reducing bias in critical areas while inducing minimal "collatoral damage" from overfitting in other areas.

- **Bayesian Occam's razor** (with **Bayes factors**) does not attempt to optimize the **bias-variance tradeoff** in this "overfitting" manner; rather, it penalizes "prior misspecifications" that are increasingly unavoidable in higher dimensions and can rapidly overwhelm the potential beneficial increases in model flexibility observable in the likelihood. 

信息准则 (Information Criterion) 指的是每增加一个参数，期望减少 $-2\log f(y | \theta)$ 大约 1 个单位，因为负对数似然越小代表模型拟合越好。DIC、AIC 和 BIC 都是通过不同方式权衡模型的拟合度与复杂性，DIC 用的是 $-2\log(p(y|\bar{\theta})) + 2p_D$，AIC 用的是 $-2\log(p(y|\hat{\theta})) + 2p$，BIC 用的是 $-2\log(p(y|\hat{\theta})) + p\ln(n)$。

Bayesian Occam's razor (Bayes factors) 偏好简单模型，但它更关注高维空间中不可避免的 prior misspecifications (先验误差)，而非样本外预测性能；相反，DIC 和 AIC 更倾向于通过校正偏差来提高样本外预测准确性。

### Widely Applicable Information Criterion (WAIC)<br> and Leave-One-Out Cross Validation (LOO-CV) 

- ***BIC:*** $\;-2\log(p(y|\hat{\theta})) + p \ln(n)\;$ is based on approximating ***Bayes factors***
    - but it's not actualy a **Bayesian** method since it doesn't integrate over the **posterior uncertainty**
- **DIC:** $\;-2\log(p(y|\bar{\theta})) + 2p_D\;$ is similarly not **fully Bayesian**
    - since $p_D \color{gray}{= {\overline{D(\theta)} - D(\bar{\theta})}} = \color{navy}{2(\log(p(y|\bar{\theta})) - \overline{\log(p(y|\theta))})}$ 
    - **conditions** on the **posterior mean** $\bar{\theta}$ rather than integrating over the **posterior uncertainty**(!)
    - The alternative $\color{purple}{\;p_D = 2\text{Var}_{p(\theta|y)}[\log(p(y|\theta))]\;}$ was unstable but it was **fully Bayesian**...

**WAIC** uses the **log pointwise predictive density** $llpd = \log\left(\prod_{i=1}^n p(y_i|\theta)\right)$ to estimate **effective model size** as

$$\color{purple}{\;p_{\text{WAIC2}} = \sum_{i=1}^n\text{Var}_{p(\theta|y)}[\log(p(y_i|\theta))]\quad \color{gray}{\text{(although note there's no longer a factor of $2$ [proof not shown])}}}$$
which provide stable estimation, and is preferred over the (also **fully Bayeisan**) $$\;\color{navy}{p_{\text{WAIC1}} = 2\sum_{i=1}^n\left(\log\left( \frac{1}{T}\sum_{t=1}^T p(y_i|\theta^{(t)}) \right) - \frac{1}{T}\sum_{t=1}^T\log(p(y_i|\theta^{(t)}))\right)\;}$$ because $p_{\text{WAIC2}}$ <u>is theoretically and empirically more similar to a **LOO-CV**</u> calculation than $p_{\text{WAIC1}}$ <font style='color:gray'></font>

BIC 和 DIC 都不是完全 Bayesian 的，因为它们没有对 posterior uncertainty (后验不确定性) 进行积分。BIC 基于近似 Bayes factors，DIC 通过计算 $p_D$ 用 posterior mean $\bar{\theta}$ 来衡量模型复杂性，而不是完全积分。

WAIC (Widely Applicable Information Criterion) 是一种更稳定的完全 Bayesian 方法，通过计算 log pointwise predictive density (llpd) 并估计模型的有效参数数量 $p_{\text{WAIC2}} = \sum_{i=1}^n \text{Var}{p(\theta|y)}[\log(p(y_i|\theta))]$，其效果与 LOO-CV (Leave-One-Out Cross Validation) 更加接近。相比之下，$p{\text{WAIC1}}$ 虽然也是完全 Bayesian 的，但在理论和实证中表现较差。

The above **information criterion** can now be extended with 

$\quad\quad\text{WAIC:} \;-2\sum_{i=1}^n\left(\log\overline{p(y_i|\theta)}\right) + 2p_{\text{WAIC2}} =  -2\sum_{i=1}^n\left(\log\left( \frac{1}{T}\sum_{t=1}^T p(y_i|\theta^{(t)}) \right)\right) + 2p_{\text{WAIC2}}$

---

The difference bewteen **WAIC** and **DIC** is that **WAIC** fully integrates over the posterior while **DIC** does not and instead conditions on the **posterior parameter mean**

- They both are "more Bayesian" than the **AIC** in incorporate the **prior** into **effective model size** estimation
- and in the same way both are "more Bayesian" than the **BIC** even though it approximates **Bayes factors**

> <font style='color:navy'>And anyway remember that **BIC** does not estimate ["out-of-sample-prediction error using a bias-corrected adjustment of within-sample error"](http://www.stat.columbia.edu/~gelman/research/published/waic_understand3.pdf)</font>


But ***WAIC*** is just a computationally tractable alternative to  
***bias corrected <u>log pointwise predictive density</u> $lppd$ Leave-One-Out Cross Validation (LOO-CV)***

$$\overset{\text{corrected}}{lppd_{loo-cv}}=\sum_{i=1}^n\log \underbrace{\left(\frac{1}{T}\sum_{t=1}^T p(y_i|\theta^{(t|-i)})\right)}_{\theta^{(t|-i)} \sim p(\theta|y_{-i})} + \underset{\text{due to using $n-1$ not $n$}}{\underbrace{lppd-\overline{lppd^{(-i)}}}_{\text{underestimated accuracy}}}$$

with ***AIC***, ***DIC*** and ***WAIC*** are asymptotically equivalent to ***log pointwise predictive density LOO-CV*** under various conditions...

WAIC (Widely Applicable Information Criterion) 与 DIC 的区别在于，WAIC 完全对 posterior (后验分布) 进行积分，而 DIC 仅依赖于 posterior parameter mean (后验均值)。两者相比，WAIC 更符合完整的 Bayesian 原则。

WAIC 的公式：$-2\sum_{i=1}^n\left(\log\left( \frac{1}{T}\sum_{t=1}^T p(y_i|\theta^{(t)}) \right)\right) + 2p_{\text{WAIC2}}$。它与 DIC 和 AIC 一样，通过引入 prior (先验信息) 来估计模型的有效参数数量。

LOO-CV (Leave-One-Out Cross Validation) 提供了更精确的样本外预测性能估计，通过计算每次移除一个数据点后得到的 log pointwise predictive density (lppd)。WAIC 只是一个计算上更可行的近似方法，与 LOO-CV 在大样本情况下渐进等价。


## Week 11 Homework (9)

### Q1: copulas

1. ~~Use the example copula code below to provide posterior inference on the dependency structure between for **your own non normally distributed data that you find**~~
2. Repeat the exercise using instead a two pass approach in the manner of https://www.pymc.io/projects/examples/en/latest/howto/copula-estimation.html
3. Describe what a copula is and how the two verions of code implement it 
4. Describe how to use this to create arbitrary multivariate GLM regressions

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

p = 3
#Psi = np.eye(p)
#a_cov = stats.invwishart(df=p+2, scale=Psi).rvs()
a_cor = (np.ones((p,p))*9+np.diag((1,1,1)))/10
a_cor[0,-1] -= 0.15
a_cor[-1,0] -= 0.15

n = 100
x = stats.multivariate_normal(mean=np.zeros(p), cov=a_cor).rvs(size=n)
plt.imshow(a_cor)
plt.colorbar();


In [ ]:
import seaborn
import pandas as pd
seaborn.pairplot(pd.DataFrame(x),height=1.5);

In [ ]:
y = x.copy()
y[:,0] = stats.gamma(a=5).ppf(stats.norm().cdf(x[:,0]))
y[:,1] = stats.expon(scale=1).ppf(stats.norm().cdf(x[:,1]))
y[:,2] = stats.chi2(df=10).ppf(stats.norm().cdf(x[:,2]))

seaborn.pairplot(pd.DataFrame(y),height=1.5);

In [ ]:
import pymc as pm
import arviz as az

with pm.Model() as copula:

    p0 = pm.HalfNormal('p0', sigma=10)
    y0 = pm.Gamma('y0', alpha=p0, beta=1, observed=y[:,0:1])
    y0_ = pm.Deterministic('y0_', 
            pm.Normal.icdf(
               pm.math.exp(pm.Gamma.logcdf(y0, alpha=p0, inv_beta=1)), 
                           mu=0, sigma=1))
    
    p1 = pm.HalfNormal('p1', sigma=10)
    y1 = pm.Exponential('y1', lam=p1, observed=y[:,1:2])
    y1_ = pm.Deterministic('y1_', 
            pm.Normal.icdf(
               pm.math.exp(pm.Exponential.logcdf(y1, mu=p1)), 
                           mu=0, sigma=1))

    p2 = pm.HalfNormal('p2', sigma=10)
    y2 = pm.ChiSquared('y2', nu=p2, observed=y[:,2:3])
    y2_ = pm.Deterministic('y2_', 
            pm.Normal.icdf(
               pm.math.exp(pm.ChiSquared.logcdf(y2, nu=p2)), 
                           mu=0, sigma=1))
    
    L,R,stds = pm.LKJCholeskyCov("R", n=3, eta=2.0, 
                                 sd_dist=pm.Exponential.dist(1.0, shape=3), 
                                 compute_corr=True)
    
    potential = pm.Potential("MVNeval", 
                             pm.logp(pm.MvNormal.dist(mu=0, cov=R),
                             pm.math.concatenate([y0_,y1_,y2_], axis=1)))
    
    idata = pm.sample()

# This probally produces a lot of warnings but it will run and provide inference


/Users/scottschwartz/miniconda3/envs/PyMC/lib/python3.11/site-packages/pytensor/tensor/rewriting/elemwise.py:691: UserWarning: Optimization Warning: The Op erfcinv does not provide a C implementation. As well as being potentially slow, this also disables loop fusion.
  warn(
/Users/scottschwartz/miniconda3/envs/PyMC/lib/python3.11/site-packages/pytensor/tensor/rewriting/elemwise.py:691: UserWarning: Optimization Warning: The Op erfcinv does not provide a C implementation. As well as being potentially slow, this also disables loop fusion.
  warn(
/Users/scottschwartz/miniconda3/envs/PyMC/lib/python3.11/site-packages/pytensor/tensor/rewriting/elemwise.py:691: UserWarning: Optimization Warning: The Op erfcinv does not provide a C implementation. As well as being potentially slow, this also disables loop fusion.
  warn(
/Users/scottschwartz/miniconda3/envs/PyMC/lib/python3.11/site-packages/pytensor/tensor/rewriting/elemwise.py:691: UserWarning: Optimization Warning: The Op erfcinv does not

In [ ]:
# Estimation is essentially correct
az.plot_trace(idata, var_names=['p0','p1','p2'])
plt.tight_layout()

In [ ]:
# Estimation is essentially correct
fig,ax = plt.subplots(1,4,figsize=(10,2))
for i,c in enumerate(list(idata.posterior['R_corr'].values.mean(axis=1))):
    ax[i].imshow(c)
    for k in range(3):
        for j in range(3):
            ax[i].text(j,k,s=str(round(c[j,k],3)), 
                       color='w', va='center', ha='center')

In [ ]:
# Q1

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

# First pass: Estimate marginal parameters for each variable

# Model for marginal 1 (Gamma)
with pm.Model() as marg1:
    p0 = pm.HalfNormal('p0', sigma=10)
    y0 = pm.Gamma('y0', alpha=p0, beta=1, observed=y[:,0])
    idata_marg1 = pm.sample(1000, tune=1000, cores=1, progressbar=False)
    
# Model for marginal 2 (Exponential)
with pm.Model() as marg2:
    p1 = pm.HalfNormal('p1', sigma=10)
    y1 = pm.Exponential('y1', lam=p1, observed=y[:,1])
    idata_marg2 = pm.sample(1000, tune=1000, cores=1, progressbar=False)
    
# Model for marginal 3 (ChiSquared)
with pm.Model() as marg3:
    p2 = pm.HalfNormal('p2', sigma=10)
    y2 = pm.ChiSquared('y2', nu=p2, observed=y[:,2])
    idata_marg3 = pm.sample(1000, tune=1000, cores=1, progressbar=False)

# For each marginal, compute the estimated CDF using the posterior means
p0_mean = az.summary(idata_marg1, var_names=['p0'])['mean'].values[0]
p1_mean = az.summary(idata_marg2, var_names=['p1'])['mean'].values[0]
p2_mean = az.summary(idata_marg3, var_names=['p2'])['mean'].values[0]

# Transform the observed data to the standard normal scale (pseudo-observations)
# For Gamma, Exponential, and ChiSquared respectively.
u0 = stats.gamma.cdf(y[:,0], a=p0_mean, scale=1)  
z0 = stats.norm.ppf(u0)

u1 = stats.expon.cdf(y[:,1], scale=1/p1_mean)
z1 = stats.norm.ppf(u1)

u2 = stats.chi2.cdf(y[:,2], df=p2_mean)
z2 = stats.norm.ppf(u2)

# Combine the pseudo-observations into a single matrix
Z = np.column_stack([z0, z1, z2])

# Second pass: Estimate the correlation (copula) using the pseudo-observations
with pm.Model() as copula_2pass:
    # LKJCholeskyCov for the 3-dimensional correlation matrix
    L, R, _ = pm.LKJCholeskyCov("R", n=3, eta=2.0, 
                                sd_dist=pm.Exponential.dist(1.0, shape=3),
                                compute_corr=True)
    # Likelihood using the multivariate normal copula on the transformed data
    # Here, each row of Z is modeled as coming from a MVN with zero mean and correlation R.
    pm.MvNormal("Z_obs", mu=np.zeros(3), chol=L, observed=Z)
    
    idata_2pass = pm.sample(1000, tune=1000, cores=1, progressbar=False)

# Plot the trace of the estimated correlation matrix elements
az.plot_trace(idata_2pass, var_names=['R_corr'])
plt.tight_layout()
plt.show()


A copula is a function that links univariate marginal distribution functions to form a full multivariate distribution, thereby separating the modeling of marginals from the dependency structure. In the one-pass approach, the model jointly estimates both the marginal parameters and the latent correlation (via deterministic transformations within a single model). In the two-pass approach above, the marginals are estimated first and used to transform the observed data to a common (Gaussian) scale; then, a separate model estimates the dependency structure using an LKJ prior on the correlation matrix, effectively implementing a Gaussian copula.

By using copulas, you can decouple the specification of marginal models from their dependency structure. This means that for multivariate generalized linear models (GLMs) with different outcome types, you can:

- Model each response variable with its own GLM (using the appropriate link functions and distributions).

- Transform the residuals or predictions of each GLM to a common scale (e.g., via the inverse CDF to obtain uniform or normal scores).

- Use a copula (e.g., a Gaussian copula with an LKJ prior on the correlation matrix) to model the joint dependency among these transformed outcomes.

This approach allows you to flexibly build complex multivariate models that capture both individual response behavior and their interdependencies, while leveraging the full Bayesian framework for uncertainty quantification.

### Q2: Variable Selection using Spike and Slab

Perform multivarite regression (or multivariate probit classification) with spike and slab variable selection priors and compare inference to analagous inference with diffuse normal priors (imposing minimal L2 style regularization on the likelihood).

You may artificially limit the size of your data to reduce the computational demands, but if you do so, discuss the behavior of the computational demands with respect to the number of observations $n$, the number of random variables $m$ making up the multivariate observations, and the number of columns of the design matrix $p$.



In [ ]:
# Q2

import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# -------------------------------
# Generate synthetic data
# -------------------------------
np.random.seed(123)
n = 100   # number of observations
p = 10    # number of predictors (columns of X)
m = 3     # number of response variables

# Design matrix X with n observations and p predictors
X = np.random.normal(0, 1, (n, p))

# True coefficients: many are zero, few nonzero (for variable selection)
true_b = np.zeros((p, m))
true_b[1, 0] = 2.0
true_b[3, 1] = -3.0
true_b[5, 2] = 1.5

# Generate responses Y with noise
Y = X @ true_b + np.random.normal(0, 1, (n, m))

# -------------------------------
# Spike-and-Slab Model
# -------------------------------
with pm.Model() as spike_slab_model:
    # Gamma: inclusion indicators for each coefficient (p x m matrix)
    gamma = pm.Bernoulli('gamma', p=0.5, shape=(p, m))
    
    # Slab: diffuse Normal priors for coefficients when included
    b_slab = pm.Normal('b_slab', mu=0, sigma=10, shape=(p, m))
    
    # Spike-and-Slab: if gamma==0, coefficient is 0; otherwise use b_slab.
    b = pm.Deterministic('b', gamma * b_slab)
    
    # Noise standard deviation for each response
    sigma = pm.HalfNormal('sigma', sigma=1, shape=m)
    
    # Linear predictor (n x m) for the m responses
    mu = pm.math.dot(X, b)
    
    # Likelihood: assume independent normal errors for each response
    Y_obs = pm.Normal('Y_obs', mu=mu, sigma=sigma, observed=Y)
    
    idata_spike = pm.sample(1000, tune=1000, cores=1, random_seed=123)

# -------------------------------
# Diffuse Normal Priors Model (minimal L2 regularization)
# -------------------------------
with pm.Model() as diffuse_model:
    # Use diffuse normal priors with large variance for coefficients
    b_diffuse = pm.Normal('b_diffuse', mu=0, sigma=1000, shape=(p, m))
    
    sigma_diffuse = pm.HalfNormal('sigma_diffuse', sigma=1, shape=m)
    
    mu_diffuse = pm.math.dot(X, b_diffuse)
    
    Y_obs_diffuse = pm.Normal('Y_obs_diffuse', mu=mu_diffuse, sigma=sigma_diffuse, observed=Y)
    
    idata_diffuse = pm.sample(1000, tune=1000, cores=1, random_seed=123)

# -------------------------------
# Compare results: Trace plots for key parameters
# -------------------------------
az.plot_trace(idata_spike, var_names=['b', 'gamma'])
plt.tight_layout()
plt.show()

az.plot_trace(idata_diffuse, var_names=['b_diffuse'])
plt.tight_layout()
plt.show()


### Q3 Variable Selection

Perform multivarite regression (or multivariate probit classification) with the horseshoe variable selection prior and compare inference to analagous inference with spike and slab priors.

The horseshoe variable selection prior is introduced here
- https://www.pymc.io/projects/docs/en/v5.6.0/learn/core_notebooks/pymc_overview.html
- and searches for "horseshoe prior pymc" on google produce additional examples



### The Horseshoe prior

The [PyMC overview](https://www.pymc.io/projects/docs/en/stable/learn/core_notebooks/pymc_overview.html) and [many](https://www.google.com/search?q=pymc+horseshoe&oq=pymc+horseshoe) other [resources](https://mellorjc.github.io/HorseshoePriorswithpymc3.html) provide ***Horseshoe prior*** [[1]](https://www.jstor.org/stable/25734098) [[2]](https://faculty.mccombs.utexas.edu/carlos.carvalho/Carvalhoetal2009.pdf) implementations 

| Half-Cauchy $\text{HC}_+(\xi)$ | Horseshoe Prior $\text{HSP}$ | Shrinkage $\kappa$ | Change of Variables|
|:-:|:-:|:-:|:-:|
|$$f(x \mid \xi) = \frac{2\cdot 1_{[x>=0]}(x)}{\pi \xi \left[1 + \left(\frac{x}{\xi}\right)^2\right]}$$|\begin{align*}w_i|\tau &\sim N(0, \sigma^{2}=\lambda_i^2\tau^2)\\\lambda_i &\sim HC_+(1)\\\tau &\sim HC_+(\tau_0)\end{align*}|\begin{align*}\kappa_{\lambda_i} ={}& 1/(1+\lambda_i^2)\\\lambda_i ={}& \sqrt{1/\kappa_{\lambda_i}-1}\\J_{\kappa_{\lambda_i}} ={}& \frac{1}{2}(\kappa_{\lambda_i}^{-1}-1)^{-\frac{1}{2}}\times \kappa_{\lambda_i}^{-2} \end{align*}|\begin{align*}f(\kappa_{\lambda_i}) = {} & f\left(\lambda_i = \sqrt{1/\kappa_{\lambda_i}-1}\right)\\ {} & \times \underbrace{\frac{1}{2}(\kappa_{\lambda_i}^{-1}-1)^{-\frac{1}{2}}\times \kappa_{\lambda_i}^{-2}}_{J_{\kappa_{\lambda_i}}} \end{align*}|



In [ ]:
fig,ax = plt.subplots(1,2,figsize=(10,3))
support = np.linspace(0,5,1000)
# shrnk = trans(spprt) = 1/(1+sprt**2)
shrinkage = 1/(1+support**2)
ax[0].plot(support, shrinkage)
ax[0].set_ylabel("Shrinkage")
ax[0].set_xlabel("$\\lambda_i$") 
# change of variables: 
# spprt = (1/shrnk-1)**0.5; E.g., 1/(1+.5**2), (1/.8-1)**0.5;
# jacobian: .5(1/shrnk-1)**(-.5)*shrnk**(-2)
shrinkage = np.linspace(0.01,.99,99)
ax[1].plot(shrinkage, 
           stats.halfcauchy(scale=1).pdf((1/shrinkage-1)**0.5) * 
           .5*(1/shrinkage-1)**(-.5)*shrinkage**(-2))
ax[1].set_title('The Horseshoe!')
ax[1].set_xlabel("Shrinkage");

In [ ]:
# Q3

import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# -------------------------------
# Generate synthetic data
# -------------------------------
np.random.seed(123)
n = 100   # number of observations
p = 10    # number of predictors (columns of X)
m = 3     # number of response variables

# Design matrix X with n observations and p predictors
X = np.random.normal(0, 1, (n, p))

# True coefficients: many are zero, few nonzero (for variable selection)
true_b = np.zeros((p, m))
true_b[1, 0] = 2.0
true_b[3, 1] = -3.0
true_b[5, 2] = 1.5

# Generate responses Y with noise
Y = X @ true_b + np.random.normal(0, 1, (n, m))

# -------------------------------
# Horseshoe Prior Model
# -------------------------------
with pm.Model() as horseshoe_model:
    # Global scale for all coefficients
    tau0 = 1.0
    tau = pm.HalfCauchy('tau', beta=tau0)
    
    # Local scale for each coefficient (p x m matrix)
    lam = pm.HalfCauchy('lam', beta=1, shape=(p, m))
    
    # Horseshoe prior for regression coefficients: 
    # each coefficient beta ~ Normal(0, tau * lam)
    beta_hs = pm.Normal('beta_hs', mu=0, sigma=tau * lam, shape=(p, m))
    
    # Noise standard deviations for each response
    sigma = pm.HalfNormal('sigma', sigma=1, shape=m)
    
    # Linear predictor for the m responses
    mu = pm.math.dot(X, beta_hs)
    
    # Likelihood: independent normal errors for each response
    Y_obs = pm.Normal('Y_obs', mu=mu, sigma=sigma, observed=Y)
    
    idata_hs = pm.sample(1000, tune=1000, cores=1, random_seed=123)

# -------------------------------
# Spike-and-Slab Model (from Q2)
# -------------------------------
with pm.Model() as spike_slab_model:
    # Inclusion indicators for each coefficient (p x m)
    gamma = pm.Bernoulli('gamma', p=0.5, shape=(p, m))
    
    # Slab: diffuse normal priors for coefficients when included
    b_slab = pm.Normal('b_slab', mu=0, sigma=10, shape=(p, m))
    
    # Spike-and-slab: if gamma==0 then coefficient is 0, else b_slab is used.
    beta_ss = pm.Deterministic('beta_ss', gamma * b_slab)
    
    sigma_ss = pm.HalfNormal('sigma_ss', sigma=1, shape=m)
    mu_ss = pm.math.dot(X, beta_ss)
    Y_obs_ss = pm.Normal('Y_obs_ss', mu=mu_ss, sigma=sigma_ss, observed=Y)
    
    idata_ss = pm.sample(1000, tune=1000, cores=1, random_seed=123)

# -------------------------------
# Compare Inference: Trace Plots
# -------------------------------
az.plot_trace(idata_hs, var_names=['beta_hs', 'tau', 'lam'])
plt.tight_layout()
plt.show()

az.plot_trace(idata_ss, var_names=['beta_ss', 'gamma'])
plt.tight_layout()
plt.show()

